# 01 — EDA

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/vod").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/kaggle/working/repo")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold

from vod import CLASSES, SEED
from vod.data import find_data_root, load_coco

DATA = find_data_root()
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
OUT.mkdir(parents=True, exist_ok=True)

## Load annotations

In [ ]:
splits, offset = {}, 0
for split in ["train", "val"]:
    images, anns = load_coco(DATA / split)
    images["split"], anns["split"] = split, split
    # ids restart in every split
    images["id"] += offset
    anns["image_id"] += offset
    offset = images["id"].max() + 1
    splits[split] = (images, anns)

images = pd.concat([s[0] for s in splits.values()], ignore_index=True)
anns = pd.concat([s[1] for s in splits.values()], ignore_index=True)
objs = anns[anns["iscrowd"] == 0].copy()
ignore = anns[anns["iscrowd"] == 1].copy()
objs["class"] = objs["cls"].map(dict(enumerate(CLASSES)))
objs["side"] = np.sqrt(objs["area"])

overview = images.groupby("split").agg(clips=("video", "nunique"), frames=("id", "size"))
overview["objects"] = objs.groupby("split").size()
overview["ignored_regions"] = ignore.groupby("split").size()
overview["objects_per_frame"] = overview["objects"] / overview["frames"]
overview

In [ ]:
test = pd.read_csv(DATA / "test_images.csv")
res = pd.concat([
    images.groupby(["split", "width", "height"])["video"].nunique().rename("clips"),
    test.assign(split="test").groupby(["split", "width", "height"])["clip"].nunique().rename("clips"),
])
res.to_frame()

## Classes

In [ ]:
counts = objs.groupby(["class", "split"]).size().unstack().reindex(CLASSES)
share = counts / counts.sum()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
counts.plot.bar(ax=ax[0], logy=True, title="Objects per class")
share.plot.bar(ax=ax[1], title="Class share")
plt.tight_layout()
counts.assign(train_share=share["train"].round(3), val_share=share["val"].round(3))

In [ ]:
per_clip = objs.groupby(["video", "class"]).size().unstack(fill_value=0)[CLASSES]
per_clip = per_clip.div(per_clip.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(per_clip.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=45)
ax.set_yticks(range(len(per_clip)), per_clip.index, fontsize=6)
ax.set_title("Class share per clip")
plt.colorbar(im)
plt.tight_layout()

## Object sizes

In [ ]:
objs["bucket"] = pd.cut(objs["area"], [0, 32 ** 2, 96 ** 2, np.inf], labels=["small", "medium", "large"])
display(pd.crosstab(objs["class"], objs["bucket"], normalize="index").reindex(CLASSES).round(3))
print(objs["bucket"].value_counts(normalize=True).round(3))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
bins = np.logspace(0, np.log10(objs["side"].max()), 60)
for c in CLASSES:
    ax[0].hist(objs.loc[objs["class"] == c, "side"], bins=bins, histtype="step", label=c)
ax[0].set(xscale="log", xlabel="sqrt(area), px", title="Object size by class")
ax[0].legend()

long_side = objs[["width", "height"]].max(axis=1)
for imgsz in [640, 1024, 1280, 1536]:
    ax[1].hist(objs["side"] * imgsz / long_side, bins=np.logspace(-0.5, 3, 60), histtype="step", label=f"imgsz {imgsz}")
ax[1].axvline(8, ls="--", c="k")
ax[1].set(xscale="log", xlabel="sqrt(area) after resize, px", title="Object size at network input")
ax[1].legend()
plt.tight_layout()

for imgsz in [640, 1024, 1280, 1536]:
    print(f"imgsz {imgsz}: {(objs['side'] * imgsz / long_side < 8).mean():.1%} of objects below 8 px")

In [ ]:
order = objs.groupby("video")["side"].median().sort_values().index
fig, ax = plt.subplots(figsize=(14, 4))
ax.boxplot([objs.loc[objs["video"] == v, "side"] for v in order], showfliers=False)
ax.set_xticks(range(1, len(order) + 1), order, rotation=90, fontsize=6)
ax.set(yscale="log", ylabel="sqrt(area), px", title="Object size per clip")
plt.tight_layout()

## Box shapes (k-means at 1280)

In [ ]:
wh = objs[["w", "h"]].to_numpy() * 1280 / long_side.to_numpy()[:, None]
km = KMeans(n_clusters=9, n_init=10, random_state=SEED).fit(np.log(wh))
order = np.argsort(km.cluster_centers_.sum(axis=1))
anchor_table = pd.DataFrame(np.exp(km.cluster_centers_[order]), columns=["w", "h"]).round(1)
anchor_table["h/w"] = (anchor_table["h"] / anchor_table["w"]).round(2)
anchor_table["share"] = (np.bincount(km.labels_)[order] / len(wh)).round(3)
anchor_table

In [ ]:
objs["aspect"] = objs["h"] / objs["w"]
objs.groupby("class")["aspect"].describe().reindex(CLASSES).round(2)

## Objects per frame

In [ ]:
images["n_objects"] = objs.groupby("image_id").size().reindex(images["id"], fill_value=0).to_numpy()
print(images["n_objects"].describe().round(1))
print("frames with > 100 objects:", (images["n_objects"] > 100).mean().round(3),
      "| max:", images["n_objects"].max())

order = images.groupby("video")["n_objects"].median().sort_values().index
fig, ax = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [1, 3]})
ax[0].hist(images["n_objects"], bins=60)
ax[0].set(xlabel="objects per frame")
ax[1].boxplot([images.loc[images["video"] == v, "n_objects"] for v in order], showfliers=False)
ax[1].set_xticks(range(1, len(order) + 1), order, rotation=90, fontsize=6)
ax[1].set(title="Objects per frame, per clip")
plt.tight_layout()

## Occlusion / truncation

In [ ]:
display(pd.crosstab(objs["class"], objs["occlusion"], normalize="index").reindex(CLASSES).round(3))
pd.crosstab(objs["class"], objs["truncation"], normalize="index").reindex(CLASSES).round(3)

## Ignored regions

In [ ]:
ignore["frame_share"] = ignore["area"] / (ignore["width"] * ignore["height"])
frame_ignore = ignore.groupby("image_id")["frame_share"].sum().clip(upper=1)
print(f"frames with ignored regions: {images['id'].isin(frame_ignore.index).mean():.1%}")
print(f"ignored share of frame: median {frame_ignore.median():.3f}, max {frame_ignore.max():.3f}")
ignore.groupby("video").size().describe().round(1)

In [ ]:
def overlap_with_ignore(objs, ignore):
    out = pd.Series(0.0, index=objs.index)
    ign = ignore.groupby("image_id")[["x", "y", "w", "h"]].apply(lambda g: g.to_numpy())
    for image_id, group in objs[objs["image_id"].isin(ign.index)].groupby("image_id"):
        o, r = group[["x", "y", "w", "h"]].to_numpy(), ign[image_id]
        ix = np.clip(np.minimum(o[:, None, 0] + o[:, None, 2], r[None, :, 0] + r[None, :, 2])
                     - np.maximum(o[:, None, 0], r[None, :, 0]), 0, None)
        iy = np.clip(np.minimum(o[:, None, 1] + o[:, None, 3], r[None, :, 1] + r[None, :, 3])
                     - np.maximum(o[:, None, 1], r[None, :, 1]), 0, None)
        out[group.index] = np.clip((ix * iy).sum(axis=1) / (o[:, 2] * o[:, 3]), 0, 1)
    return out


objs["ignore_overlap"] = overlap_with_ignore(objs, ignore)
print(f"objects >= 50% inside ignored region: {(objs['ignore_overlap'] >= 0.5).mean():.2%}")

## Same-track IoU between frame t and t+k

In [ ]:
def box_iou(a, b):
    ix = np.clip(np.minimum(a[:, 0] + a[:, 2], b[:, 0] + b[:, 2]) - np.maximum(a[:, 0], b[:, 0]), 0, None)
    iy = np.clip(np.minimum(a[:, 1] + a[:, 3], b[:, 1] + b[:, 3]) - np.maximum(a[:, 1], b[:, 1]), 0, None)
    inter = ix * iy
    return inter / (a[:, 2] * a[:, 3] + b[:, 2] * b[:, 3] - inter)


tracks = objs.dropna(subset=["track_id"])[["video", "track_id", "frame_id", "x", "y", "w", "h"]]
rows = []
for k in [1, 2, 3, 5, 10, 15, 20, 30]:
    nxt = tracks.assign(frame_id=tracks["frame_id"] - k)
    m = tracks.merge(nxt, on=["video", "track_id", "frame_id"], suffixes=("", "_k"))
    iou = box_iou(m[["x", "y", "w", "h"]].to_numpy(), m[["x_k", "y_k", "w_k", "h_k"]].to_numpy())
    rows.append({"step": k, "median_iou": np.median(iou), "share_iou_gt_0.7": (iou > 0.7).mean()})
redundancy = pd.DataFrame(rows).round(3)
redundancy.plot(x="step", marker="o")
redundancy

## GT on sample frames (green — objects, red — ignored)

In [ ]:
def frame_path(row):
    for base in [DATA / row["split"], DATA, DATA / row["split"] / "images"]:
        if (base / row["file_name"]).exists():
            return base / row["file_name"]
    raise FileNotFoundError(row["file_name"])


def show_frame(row, ax):
    ax.imshow(Image.open(frame_path(row)))
    for o in objs[objs["image_id"] == row["id"]].itertuples():
        ax.add_patch(Rectangle((o.x, o.y), o.w, o.h, fill=False, ec="lime", lw=0.8))
        ax.text(o.x, o.y, o.cls, color="yellow", fontsize=5)
    for r in ignore[ignore["image_id"] == row["id"]].itertuples():
        ax.add_patch(Rectangle((r.x, r.y), r.w, r.h, fill=False, ec="red", hatch="//", lw=1))
    ax.set_title(f"{row['video']} #{row['frame_id']} ({row['n_objects']} obj)", fontsize=8)
    ax.axis("off")


sample = images[images["id"].isin(ignore["image_id"])].sample(3, random_state=SEED)
sample = pd.concat([sample, images.nlargest(1, "n_objects"), images.sample(2, random_state=SEED)])
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for (_, row), ax in zip(sample.iterrows(), axes.flat):
    show_frame(row, ax)
plt.tight_layout()

## Folds by clip

In [ ]:
images["fold"] = -1
for fold, (_, idx) in enumerate(GroupKFold(n_splits=5).split(images, groups=images["video"])):
    images.loc[images.index[idx], "fold"] = fold
clips = images.groupby("video").agg(split=("split", "first"), fold=("fold", "first"),
                                    frames=("id", "size")).reset_index()
clips.to_csv(OUT / "folds.csv", index=False)
clips.groupby("fold").agg(clips=("video", "size"), frames=("frames", "sum"))

## Observations